# SkillRadar – Del 3: Skill extraction, matching, LLM og hybridmodeller
## Meget detaljeret eksamensnotebook

**Fokus:** Hvad har I faktisk implementeret, hvordan fungerer hver metode, hvorfor blev den valgt, hvilke output giver den, og hvilke alternativer kunne I have brugt?

**Kilder (læst i GitHub 5. november 2026):** [`matcher.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/matcher.py), [`variants.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/variants.py), [`llm_matcher.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/llm_matcher.py), [`evaluate.py`](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/evaluate.py), [`llm_frozen_settings.txt`](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/llm_frozen_settings.txt), projektets `README.md`.

**Tre slags indhold:**
1. **Faktisk projektkode:** direkte uddrag eller tro gengivelse af funktionalitet fra repository.
2. **Kørbare undervisningseksempler:** små, kunstige CV'er/jobopslag og en forenklet matcher. De er *ikke* projektets faktiske målte resultater.
3. **Dokumenterede forsøgsresultater:** tal fra projektets README. Vi opfinder ikke nye testresultater.

Kørbare celler kræver kun Python og `pandas` (som I allerede bruger). Nogle valgfrie celler kan benytte det lokale repository, hvis det ligger på maskinen sammen med denne notebook. Ingen API-nøgler eller personlige CV'er kræves.

## 1. Præcis hvad Del 3 handler om

Del 1 beskrev webappens brugerflow. Del 2 beskrev dataforberedelsen. **Del 3 er selve metoden til at omsætte ustruktureret tekst til en sammenlignelig liste over navngivne teknologier.**

Eksempel:

- CV: `I built dashboards with Power BI and analysed customer data in Python.`
- Jobopslag: `Knowledge of Microsoft Power BI, Python and SQL required.`
- Begge skal mappes til *samme kanoniske teknologier*, uanset præcis stavemåde.

Flowet er `rå tekst → tokenisering → genkendelse af aliaser → kanonisk O*NET-navn → set af teknologier → sammenligning`.

**Vigtig afgrænsning:** Jeres vocabulary handler om *named tools/technologies*. `machine learning`, `statistics`, `teamwork` og lignende generelle færdigheder er ikke del af hovedrapportens teknologimatching. Dette er et bevidst scopevalg og ikke blot en fejl i extractoren.

## 2. Hvad er faktisk i webappen, og hvad er en forskningssammenligning?

| Metode | Fil | Er den brugt i `app.py`? | Formål |
|---|---|---|---|
| Keyword/O*NET | `matcher.py` → `KeywordMatcher` | **Ja** | Hurtig, lokal ekstraktion fra CV og jobbeskrivelser |
| Exact/vendor/acronym/full | `variants.py` | Ikke som brugerens valg | Ablation: isolere bidrag fra hver normaliseringsregel |
| LLMOnly | `llm_matcher.py` | **Nej**, separat evaluering | Kan LLM finde værktøjer uden lukket kandidatordliste? |
| Hybrid | `llm_matcher.py` | **Nej**, separat evaluering | Keyword-kandidater eftertjekkes og manglende værktøjer tilføjes |
| HybridVerify | `llm_matcher.py` | **Nej**, separat evaluering | Måle isoleret effekt af kontekst-verificering |

**Forklaring til eksamen:** Appens CV-flow bruger `matcher.extract_tools` og brugerkorrektion (`human in the loop`). Selvom I har evalueret en LLM og en hybrid, betyder det ikke, at hovedappen automatisk kalder Gemini, når en bruger uploader et CV.

**Hvorfor den forskel?** En videnskabelig sammenligning skal kunne efterprøve dyrere metoder, selv når den konkrete brugerapp foreløbigt prioriterer hastighed, omkostninger og CV-privatliv.

## 3. Begreber, I skal kende

**Free text:** almindelige sætninger i CV'er og jobopslag i modsætning til strukturerede kolonner.

**Named entity extraction:** at finde navngivne elementer som `Docker`, `AWS` og `scikit-learn` i teksten.

**Entity linking / normalization:** at knytte et fundet navn til en fælles identitet, så `Power BI` og `Microsoft Power BI` betyder samme værktøj.

**Taxonomy/vocabulary:** O*NET er jeres kilde til kanoniske teknologier. I benytter primært teknologinavne og aliaser, ikke en avanceret grafbaseret hierarkisk ræsonnering i webappen.

**Closed vocabulary:** keyword-matcheren kan kun returnere kanoniske værktøjer i sit O*NET-vokabular.

**Open vocabulary:** LLM kan foreslå værktøjer, der slet ikke findes i O*NET. De kan repræsenteres som `OOV:<navn>` i evalueringen.

**False positive (FP):** Systemet påstår, at noget er et værktøj, selvom det ikke er nævnt som et værktøj. **False negative (FN):** Systemet overser et faktisk navngivet værktøj.

## 4. Hvordan bygger `KeywordMatcher` ordforrådet?

Projektfil: `skillradar/matcher.py`.

Den faktiske funktion `build_vocab(software)` grupperer O*NET-rækker pr. `Workplace Example`, markerer om en teknologi noget sted er angivet som `Hot Technology` **eller** `In Demand`, og frasorterer generiske navne via `is_generic()`.

```python
flags = software.groupby("Workplace Example")[["Hot Technology", "In Demand"]].agg(
    lambda s: (s == "Y").any()
)
important = flags.index[flags["Hot Technology"] | flags["In Demand"]]
vocab = sorted(t for t in important if not is_generic(t))
```

**Hvorfor dette?** En opgaverelevant, afgrænset værktøjsliste gør extraction reproducerbar og mindsker tilfældige hits på brede kompetencer.

**Tradeoff:** En ny teknologi, fx et brand som ikke er med i O*NET-udtrækket, kan ikke genkendes af den almindelige matcher. Den kan optræde i LLM-metodernes OOV-resultater, men ikke i hovedappens lukkede liste.

**Alternativ:** ESCO eller et domænespecifikt manuelt katalog; eller et åbent LLM-vokabular. Bredere coverage kan kræve mere efterbehandling og evaluering.

In [ ]:
from IPython.display import display
import pandas as pd
from collections import Counter

onet_demo = pd.DataFrame([
    {'Workplace Example':'Python','Hot Technology':'Y','In Demand':'N'},
    {'Workplace Example':'Python','Hot Technology':'N','In Demand':'Y'},
    {'Workplace Example':'Microsoft Power BI','Hot Technology':'N','In Demand':'Y'},
    {'Workplace Example':'Obsolete software','Hot Technology':'N','In Demand':'N'},
])
flags = onet_demo.groupby('Workplace Example')[['Hot Technology','In Demand']].agg(lambda s: (s=='Y').any())
display(flags)
print('Udvalgt før generisk-filter:', list(flags.index[flags['Hot Technology'] | flags['In Demand']]))

### Forklaring af output

`Python` findes i flere O*NET-rækker, men samles til én teknologi. `any()` ser efter mindst én `Y`; den kræver ikke, at alle tilhørende occupation-rækker er markeret. `Obsolete software` vælges ikke i illustrationen, fordi begge flags er `N`.

Den rigtige `build_vocab` har **derudover** `is_generic`; eksemplet ovenfor illustrerer kun flag-filteret. Et O*NET-flag er ikke en procentdel af jeres LinkedIn-jobopslag.

## 5. Aliasgenerering: sådan kan ét værktøj få flere navne

Projektets `aliases(name)` genererer blandt andet:
- Hele O*NET-navnet.
- Forkortede visningsnavne for særlige `KEEP`-tilfælde.
- Navnet uden et afsluttende ` software`.
- Navnet uden visse vendor-prefixes (fx `Microsoft Power BI` → `Power BI`), med undtagelser i `NO_SHORT`.
- En afsluttende forkortelse **når bogstaverne passer til den foranstående formulering**, fx `SQL` fra `Structured query language SQL`.

**Vigtigt:** Der laves ikke vilkårlige synonymrelationer, som fx at sige at PyTorch = TensorFlow. Det er relaterede men forskellige teknologier, og ville være en forkert præcis match.

**Hvorfor bruger I regler?** De er nemme at gennemskue, hurtige og fungerer uden træning eller eksterne API'er. Til gengæld opdager de ikke alle alternative formuleringer.

In [ ]:
# Pædagogisk, lukket alias-eksempel: ikke hele matcher.py
aliases_demo = {
    'Python': ['Python'],
    'Microsoft Power BI': ['Microsoft Power BI','Power BI'],
    'Structured query language SQL': ['Structured query language SQL','SQL'],
    'Amazon Web Services AWS software': ['Amazon Web Services AWS software','AWS'],
}
for canonical, names in aliases_demo.items():
    print(f'{canonical} -> {names}')

### Outputforklaring

Det venstre navn er den **kanoniske** identitet. Højre side er måder, en bruger kan skrive den på. En korrekt normalisering giver én identitet, selv hvis et opslag skriver `SQL` og et CV skriver O*NET's lange SQL-navn.

Her er aliaserne en undervisningsliste. I jeres rigtige program bygges de af `aliases()` sammen med undtagelsesregler.

## 6. Tokenisering: hvorfor `R`, `C++` og `Node.js` er vanskelige

Fra `matcher.py`:

```python
TOKEN = re.compile(r"[A-Za-z0-9+#&]+(?:[.-][A-Za-z0-9+#&]+)*")
MAX_NGRAM = 8

def tokens(text: str) -> list[str]:
    return TOKEN.findall(text)
```

**Forklaring:** `TOKEN.findall()` deler teksten i tokens, men bevarer vigtige tegn i tech-navne. Den sædvanlige `str.split()` kan være utilstrækkelig, når specialtegn eller tegnsætning er centrale for identiteten. Det gælder fx `C++`, `C#` og `Node.js`.

**N-gram:** matcher'en prøver sekvenser af 1, 2, 3, ... op til maksimalt otte tokens, fordi mange teknologier består af flere ord.

**Alternativer:** spaCy tokeniserer, men generel sprogtokenisering passer ikke nødvendigvis til teknologinavne; specialiserede regex-regler eller en NLP-model kan supplere.

In [ ]:
import re
TOKEN = re.compile(r"[A-Za-z0-9+#&]+(?:[.-][A-Za-z0-9+#&]+)*")
example = 'Experience with C++, C#, Node.js, scikit-learn, R&D and SQL.'
print(TOKEN.findall(example))
print('Alle 1- og 2-grams fra:', 'Power BI and SQL')
toks = TOKEN.findall('Power BI and SQL')
for n in (1,2):
    print(n, [' '.join(toks[i:i+n]) for i in range(len(toks)-n+1)])

### Hvad outputtet betyder

`C++` og `C#` holdes adskilt, så de ikke fejlagtigt begge reduceres til `C`; `Node.js` kan være én enhed, og `R&D` bliver ikke til selvstændig teknologien `R`.

2-grams omfatter fx `Power BI`. En ren søgning efter ét ord ad gangen ville ikke kunne identificere denne frase sikkert.

## 7. Case-sensitive matching og falske positive

I `CASE_SENSITIVE` angiver I ord, som også kan være almindelige engelske ord eller meget korte betegnelser: fx `R`, `Go`, `Excel`, `React`, `Swift` og `Spark`.

For disse aliaser kan store/små bogstaver få betydning. Derimod normaliseres mange andre navne til lowercase for at tåle forskellig kapitalisering.

**Eksempel:** `We use Go` er mere informativt om programmeringssproget end `go further`. `R` skal ikke optræde som match inde i `R&D` eller andre ord.

**Begrænsning:** Store bogstaver løser ikke kontekst helt. En overskrift i uppercase eller ordet `Go` i starten af en sætning kan stadig ramme forkert. Derfor har appen et menneskeligt valideringstrin.

**Alternativer:** POS-tags, kontekstbaseret klassifikation, en NER-model eller LLM-verificering.

In [ ]:
case_sensitive = {'Go', 'R', 'Excel'}
phrases = ['Go further and learn', 'We build services in Go', 'Excel at teamwork', 'Excel dashboards']
for phrase in phrases:
    toks = TOKEN.findall(phrase)
    hits = [w for w in toks if w in case_sensitive]
    print(f'{phrase!r}: {hits}')

**Læsevejledning:** Den lille tokenregel viser, at `Go` kan genkendes i både et ægte tech-udsagn og et engelsk udsagn. Det er en demonstration af, at *case alene ikke er nok*, ikke en påstand om at jeres matcher altid laver præcis dette output.

## 8. Hvordan mapper `KeywordMatcher` alias → kanonisk værktøj?

I konstruktøren:

```python
self.vocab, popularity = build_vocab(software)
self.alias_to_tool = {}
for tool in sorted(self.vocab, key=lambda t: popularity.get(t, 0)):
    for a in aliases(tool):
        if tokens(a):
            self.alias_to_tool[alias_key(a)] = tool
```

1. `build_vocab` giver et katalog og en `popularity`-tælling baseret på forekomster i O*NET-rækker.
2. Teknologier sorteres efter popularitet, inden alias-tabellen bygges.
3. Flere kanoniske teknologier kan dele et alias. Ved nøglekollision overskriver sidste tildeling den tidligere.
4. Derfor vinder ifølge kodekommentaren den mere populære teknologi typisk aliaset.

**Vigtigt:** Dette er en prioriteringsregel, ikke en egentlig semantisk disambiguation baseret på sætningen. Det kan give systematiske fejl for tvetydige aliaser.

**Alternativer:** Drop alle tvetydige aliaser, bevar flere kandidater med kontekstklassifikator, eller opret én entydig identifikator per tool med flere navne men separate betydninger.

## 9. Den konkrete `explain()`-algoritme

Fra `matcher.py`:

```python
def explain(self, text):
    if not isinstance(text, str):
        return {}
    toks = tokens(text)
    low = [w.lower() for w in toks]
    found = {}
    for n in range(1, self.max_n + 1):
        for i in range(len(toks) - n + 1):
            for g in (" ".join(low[i:i + n]), " ".join(toks[i:i + n])):
                if g in self.alias_to_tool:
                    found[g] = self.alias_to_tool[g]
    return found
```

**Hvad sker der?** For hvert n-gram og startposition laves en lowercase og en original-case udgave, og der søges i `alias_to_tool`. Returneringen er en ordbog med *det alias der ramte* som nøgle og *kanonisk teknologi* som værdi.

**Hvorfor to udgaver?** Lowercase understøtter normale stavemåder, original-case understøtter de følsomme navne.

**Konsekvens:** Denne metode registrerer *tilstedeværelsen* af navnet, ikke om værktøjet er anvendt af kandidaten, kun nævnt som sammenligning, eller optræder som et virksomhedsmærke. Derfor er forklarbarhed og evalueret præcision afgørende.

In [ ]:
# Lille kørbar model af samme n-gram-princip (ikke komplet matcher)
alias_to_tool_demo = {
    'python': 'Python',
    'power bi': 'Microsoft Power BI',
    'sql': 'Structured query language SQL',
    'Go': 'Go programming language',
}
def demo_explain(text):
    toks = TOKEN.findall(text); low = [w.lower() for w in toks]
    found = {}
    for n in range(1, 4):
        for i in range(len(toks)-n+1):
            for gram in (' '.join(low[i:i+n]), ' '.join(toks[i:i+n])):
                if gram in alias_to_tool_demo:
                    found[gram] = alias_to_tool_demo[gram]
    return found
text_demo = 'Built Power BI reports in Python and SQL. We also use Go.'
print('Matches:', demo_explain(text_demo))
print('Kanoniske teknologier:', set(demo_explain(text_demo).values()))

### Output og hvorfor `set` bruges

Outputtet viser aliases (`power bi`, `python`, `sql`, `Go`) og deres kanoniske O*NET-lignende navne. Når vi tager `set(...values())`, registreres teknologien kun én gang, selv om teksten omtaler den flere gange.

**I den faktiske kode** gør `extract_tools(text)` netop `set(self.explain(text).values())`. Det er et centralt led i demand-beregningen fra Del 2: Hvert jobopslag tæller højst én gang pr. teknologi.

## 10. Evidence: Hvorfor kan brugeren se *hvor* et match kom fra?

`matcher.evidence(text, window=45)` søger de samme teknologier, men bygger tekstuddrag omkring den første forekomst af hver fundet teknologi.

- Funktionen finder token-positioner i originalteksten.
- Den prøver længste n-grams først.
- Den finder start/slutpositionen for det matchende alias.
- `window=45` tager op til 45 tegn før og efter.
- Det genkendte udtryk markeres med `**...**`, som UI'et senere gør synligt.

**Hvorfor?** Brugeren kan se eksempelvis `"... dashboards with **Power BI** for ..."` og vurdere, om det er et faktisk værktøj. Det støtter både explainability og human-in-the-loop-korrektion.

**Alternativer:** Lagring af alle matches med offsets, sentence-level evidence eller visning af hele sætningen. Bedre kontekst kan koste mere UI-plads og potentielt eksponere flere CV-oplysninger på skærmen.

## 11. Manuel fjernelse af dårlige aliaser

I har i koden:

```python
MANUAL_DROP = {"Google", "Facebook"}
```

Projektkommentarerne dokumenterer, at I undersøgte virkelige tekstuddrag for disse ord. `Google` blev ofte nævnt som arbejdsgiver eller et generelt brand; `Facebook` var oftest en social kanal/virksomhed frem for teknologien, I ønskede at måle.

Til gengæld beholdt I `C` og `Go` efter manuel kontrol.

Funktionerne `alias_share()` og `suggest_drops()` opdager *mistænkeligt hyppige* aliaser, men sletter dem ikke automatisk: Hyppighed alene er ikke evidens for fejl, da fx Excel kan være meget udbredt.

**Eksamenspoint:** I har lavet en datadrevet **fejlanalyse med menneskelig kontrol**, ikke blot forsøgt at maksimere antallet af matches. Andre metoder: kontekst-regler, supervised klassifikation eller LLM-kontekstkontrol.

## 12. Matching på CV vs. matching på Kaggle-labels

I har *to* forskellige inputtyper:

1. **Fritekst**, fx `We use R for modelling`. Her bruges `extract_tools()`.
2. **Allerede udtrukne labels**, fx en liste `['python','r','scikitlearn']`. Her findes `extract_tools_from_labels()`.

I labels er navnet en selvstændig label; `r` kan derfor være R-programmeringssproget, selv om det er lowercase. Jeres `extract_tools_from_labels()` forsøger først eksakt normaliseret alias-opslag gennem `_squash_index()` og derefter fri-tekst-extraction.

`squash('scikit-learn')` fjerner tegnsætning og små/store bogstaver, så `scikitlearn` og `scikit-learn` kan sammenlignes mere robust.

**Hvorfor separat metode?** En isoleret label har en anden betydning end en enkelt bogstavforekomst i en almindelig sætning. At blande de to kan give misvisende sammenligninger.

In [ ]:
def squash_demo(text):
    return re.sub(r'[^a-z0-9+#]', '', text.lower())
for term in ['scikit-learn','scikitlearn','Power BI','power-bi','R&D','R']:
    print(f'{term:15} -> {squash_demo(term)}')

### Outputfortolkning

`scikit-learn` og `scikitlearn` får samme nøgle, ligesom variationer af Power BI. Men bemærk, at `R&D` og `R` ikke reduceres til samme nøgle. Funktionen løser **stavemåde/punktuation**, ikke alle semantiske tvetydigheder.

## 13. Ablation: Hvorfor fire varianter?

I `variants.py` er de fire **kumulative** trin:

| Variant | Hvad bliver tilføjet? | Hypotese |
|---|---|---|
| `exact` | Kun fulde O*NET-navne, case-insensitivt | Baseline: matcher stærkt formelle navne |
| `vendor` | + dropper visse vendor-prefixes og `software` | Typiske kortformer findes |
| `acronym` | + genkender begrundede forkortelser såsom `SQL`, `AWS` | Mange CV'er bruger forkortelser |
| `full` | + case-sensitive regler for korte/tvetydige navne; faktisk `KeywordMatcher` | Kan øge præcision uden for stort recall-tab |

**Det vigtige er, at alle varianter bruger samme ordforråd og samme test.** Man ændrer ét normaliseringstrin ad gangen for at måle dets effekt – det hedder en ablation study.

**Alternativer:** Ikke-kumulative forsøg, hvor alle kombinationer af regler afprøves (faktorielt design), kan vise interaktioner, men kræver mange flere målinger.

In [ ]:
# Illustration af aliastrappens effekt på et CV
text_case = 'Used SQL and Power BI for analytics.'
patterns_demo = {
    'exact': {'Structured query language SQL': 'SQL', 'Microsoft Power BI': 'Power BI'},
    'vendor': {'Structured query language SQL': 'SQL', 'Microsoft Power BI': 'Power BI', 'Power BI': 'Power BI'},
    'acronym': {'Structured query language SQL': 'SQL', 'Microsoft Power BI': 'Power BI', 'Power BI': 'Power BI', 'SQL': 'SQL'},
}
for stage, name_map in patterns_demo.items():
    found = {v for alias,v in name_map.items() if re.search(r'(?<!\w)'+re.escape(alias)+r'(?!\w)', text_case, re.I)}
    print(stage, sorted(found))

**Output:** `exact` finder typisk ikke korte betegnelser, `vendor` kan finde Power BI, og `acronym` kan også finde SQL. Dette er et lille eksempel på logikken; resultaterne er **ikke** jeres målte ablation-metrics.

## 14. De tre LLM-metoder – overblik

I `skillradar/llm_matcher.py` deler klasserne et interface:

```python
extract_tools(text: str) -> set[str]
```

Dermed kan `evaluation/evaluate.py` evaluere modeller med den samme scoringkode.

- **LLMOnly:** Gemini læser teksten og udtrækker alle navngivne værktøjer med kildecitater; ingen keyword-kandidater på forhånd.
- **Hybrid:** KeywordMatcher leverer kandidater, Gemini vurderer dem i kontekst og foreslår yderligere fund.
- **HybridVerify:** Samme Gemini-svar som Hybrid, men bruger kun verificerede keyword-kandidater, **ikke** nye værktøjer. Dermed kan I isolere værdien af verifikation fra værdien af nye fund.

Dette er **inference/prompting** på en prætrænet model. I har **ikke trænet eller finetunet** Gemini i det dokumenterede forsøg.

## 15. Hvad står der i LLM-prompten?

Jeres `RULES` beskriver værktøjer som navngivne produkter, programmeringssprog, biblioteker, frameworks, databaser, cloud services eller platforme, der faktisk indgår i arbejdet.

Prompten instruerer bl.a. modellen i:
- **Ikke** at tælle arbejdsgivere som værktøjer.
- **Ikke** at tælle generelle metoder som `statistics` eller `machine learning`.
- At angive en kort **ordret quote**, der beviser hvert fund.
- Ikke at opfinde værktøjer, der normalt hører til rollen, men ikke står i teksten.

Den ønskede form er valideret via Pydantic:

```python
class Tool(BaseModel):
    name: str
    quote: str
class Extraction(BaseModel):
    tools: list[Tool]
```

**Hvorfor structured output?** Det reducerer parse-fejl, giver samme schema for alle outputs og gør automatisk grounding/kvalitetskontrol mulig. Alternativer: JSON parsing uden schema, function calling eller frit tekstsvar; sidstnævnte er vanskeligere at evaluere pålideligt.

In [ ]:
# En syntetisk LLM-respons efter projektets schema (ingen API bruges)
fake_llm_answer = {'tools': [
    {'name':'Python', 'quote':'Built a service in Python'},
    {'name':'Docker', 'quote':'We deploy with Docker'},
    {'name':'Kubernetes', 'quote':'Kubernetes is required'},
]}
print(pd.DataFrame(fake_llm_answer['tools']).to_string(index=False))

**Outputforklaring:** Systemet kan behandle hvert værktøj separat og kontrollere, om det angivne citat faktisk findes. En model, der blot skrev `Python, Docker, Kubernetes` uden evidence, ville være sværere at kontrollere.

## 16. Grounding: Hvad gør I for at undgå hallucinationer?

Projektets `grounded(quote, text)` normaliserer typografiske citationstegn, whitespace og case og tjekker derefter, om citatet findes i originalteksten. `find_quote` bruges som fallback for teknologinavnet.

I `_link_all`:

1. Hvis LLM'ens quote findes i teksten, kan værktøjet gå videre.
2. Hvis quote ikke findes, prøver systemet om *værktøjsnavnet selv* findes som et helt ord/frase.
3. Hvis ingen af delene findes, **droppes fundet**, og `dropped_ungrounded` registreres.

**Hvorfor to veje?** LLM'en kan identificere den rigtige teknologi, men ikke kopiere den korte quote præcist. Fallback bevarer sådanne fund.

**Begrænsning:** Et ord kan være i teksten og alligevel omtale en virksomhed eller have en anden betydning. Grounding beviser **forekomst**, ikke at fortolkningen er korrekt.

In [ ]:
# En enkel grounding-illustration, inspireret af projektets kontrol
text = 'We deploy with Docker and write Python scripts.'
answers = [
    {'name':'Docker','quote':'deploy with Docker'},
    {'name':'Python','quote':'write Python scripts'},
    {'name':'Kubernetes','quote':'Kubernetes is required'},
]
def normalise_demo(s):
    return ' '.join(s.lower().split())
for a in answers:
    quote_ok = normalise_demo(a['quote']) in normalise_demo(text)
    name_ok = bool(re.search(r'(?<!\w)'+re.escape(a['name'])+r'(?!\w)', text, re.I))
    print(a['name'], 'quote:', quote_ok, 'name:', name_ok, 'keep:', quote_ok or name_ok)

**Output:** Docker og Python beholdes; Kubernetes droppes, fordi det hverken citeres eller nævnes i originalteksten. I har dermed et konkret forsvar mod, at modellen tilføjer typiske værktøjer for jobbet, som annoncen slet ikke nævner.

## 17. Linker: O*NET, embeddings og OOV

Efter LLM'en foreslår et værktøj, bruger `Linker.link(name)` følgende rækkefølge:

1. **Alias lookup:** samme normaliseringsprincip som menneskelige labels, så fx `Power BI` knyttes til kanonisk O*NET.
2. **Embedding matching:** hvis alias ikke matcher, og en embeddings-model er aktiveret, vælges den nærmeste O*NET-teknologi *kun* ved cosine similarity over threshold.
3. **OOV:** hvis ingen sikker mapping findes, returneres `OOV:<navn>`, i stedet for fejlagtigt at presse det ind i en vilkårlig O*NET-kategori.

**Hvorfor kan embedding-linking give fejl?** To værktøjer kan være semantisk beslægtede uden at være samme produkt, fx `BigQuery` vs. `MySQL` eller `Kinesis` vs. `Kubernetes`. Et højt cosine-tal betyder ikke identitet.

**Dokumenteret fra `llm_frozen_settings.txt`:** I satte `SKILLRADAR_LINK_THRESHOLD=1.0`, hvilket deaktiverer embedding-linking i den frosne evaluering, fordi aktiveret embedding-linking forringede udviklingssættets resultater.

**Vigtig pointe:** En avanceret komponent blev *fravalgt på baggrund af tests* – det er god metodekritik, ikke et nederlag.

In [ ]:
# Syntetisk illustration af hvorfor beslægtede ord ikke er identiske
fake_similarity = {'Amazon SageMaker': 0.81, 'Google BigQuery': 0.88, 'Kubernetes': 0.76}
thresholds = [0.80, 1.00]
for threshold in thresholds:
    print('threshold:', threshold)
    for name, similarity in fake_similarity.items():
        print(' ', name, '=>', 'embedding-link' if threshold < 1.0 and similarity >= threshold else 'ingen embedding-link')

**Output:** Ved threshold 0,80 bliver flere eksempler til embedding-links; ved 1,0 bliver ingen det. De syntetiske tal her er *kun* didaktiske. Det centrale er kontrollogikken, ikke de viste similarity-scores.

## 18. Hvordan virker Hybrid nøjagtigt?

Fra `Hybrid.extract_tools`:

```python
cands = self.keyword.extract_tools(text)
answer = self._ask(SYSTEM_VERIFY, ..., Verification)
by_name = {squash(v.candidate): v for v in answer.verified}
kept = set()
for c in cands:
    v = by_name.get(squash(display_name(c))) or by_name.get(squash(c))
    if v is not None and not v.keep:
        # afvises
        pass
    else:
        kept.add(c)   # hvis LLM'en ikke svarer, beholdes keyword-match
if self.add:
    kept |= self._link_all(answer.additional, text, info)
```

**Det særlige designvalg:** Hvis Gemini ikke giver en afgørelse for en keyword-kandidat, stoler systemet som default på keyword-matcheren. Kun et **eksplicit `keep=False`** fjerner kandidaten.

**Hvorfor?** Det beskytter mod at miste korrekte baseline-matches på grund af et mangelfuldt modelsvar.

**HybridVerify** arver samme algoritme, men `add=False` fjerner hele `additional`-bidraget. De to kan derfor sammenlignes direkte, uden nye API-svar eller forskellig sampling.

In [ ]:
# Syntetisk hybrid-flow: ingen API og ikke projektmåling
keyword_candidates = {'Python','Docker','Go'}
verified = {'Python': True, 'Docker': True, 'Go': False}
additional = {'Databricks'}
verified_only = {x for x in keyword_candidates if verified.get(x, True)}
hybrid = verified_only | additional
print('Keyword:', sorted(keyword_candidates))
print('HybridVerify:', sorted(verified_only))
print('Hybrid:', sorted(hybrid))

**Output:** HybridVerify fjerner `Go` på baggrund af eksplicit afvisning. Hybrid gør det samme, men tilføjer `Databricks`. Dette adskiller effekten af **kontekst-verificering** og **udvidet recall**.

## 19. Few-shot prompts: hvorfor kun dev-eksempler?

I `examples()` i `llm_matcher.py` konstruerer I user/model-eksempelpar fra allerede håndmærkede **dev**-opslag, med en eksplicit kontrol på at `split == 'dev'`. I den frosne indstilling anvender I `post_003` og `post_005`.

**Hvorfor?** Eksemplerne demonstrerer jeres definition af »tool« for en prætrænet model. De justerer prompten uden at opdatere vægte.

**Hvorfor aldrig test-eksempler?** Hvis det samme opslag både bruges som løsningsskabelon i prompten og i evalueringen, er testen forurenet (*data leakage*). De endelige testresultater ville blive optimistiske.

**Alternativer:** Zero-shot, flere dev-eksempler, retrieval-augmented few-shot eller supervised fine-tuning på et separat træningssæt. I har valgt en lille, kontrolleret few-shot opsætning.

## 20. Caching og privatliv

`_cached()` hasher kombinationen af modelnavn, promptversion, input, schema osv. til en filnøgle. API-resultatet skrives til `data/llm_cache/` og kan genbruges ved evaluering.

**Hvorfor?** API-kald er langsommere og kan koste penge. En frosset cache gør gentagne metricberegninger mere stabile og billige.

**Privatliv:** Jeres README og kode bemærker, at cachen kan indeholde citerede CV-passager og derfor er gitignored. Appens almindelige keyword-flow behøver ikke at sende CV-tekst til Gemini. Projektkoden advarer specifikt mod at sende CV'er via en gratis API-tier uden samtykke og korrekte dataforhold.

**Begrænsning:** Lokal caching kræver stadig adgangskontrol, oprydning og opmærksomhed på persondata. Et gitignore er ikke i sig selv en fuld sikkerhedsstrategi.

**Alternativer:** RAM-cache for følsomme dokumenter, krypteret korttidslagring, anonymisering eller ingen cache.

## 21. Hvorfor er resultatet ikke »vi trænede en AI-model«?

Projektets dokumenterede metoder omfatter:

- Regelbaseret keyword-/aliasmatching.
- API-inference med en prætrænet Gemini-model.
- Promptregler, few-shot eksempler fra dev og structured JSON-respons.
- Postprocessing: grounding, linking, OOV samt sammenligning mod labels.

**Det er ikke dokumenteret, at I træner eller fine-tuner modelvægtene.** En korrekt beskrivelse er: *»Vi sammenlignede en deterministisk keyword-baseline med prætrænet LLM-inference og en hybrid, der kombinerer dem.«*

**Hvorfor skellet betyder noget:** Modellen har allerede lært sprog under prætræningen, men I har ikke selv gennemført backpropagation eller parameteropdateringer i dette projekt.

## 22. Evalueringen: Hvem bestemmer hvad der er korrekt?

I `evaluation/evaluate.py` bruges menneskeligt mærkede tools (gold labels). Projektets README angiver:
- **40** data-scientist-postings blev dobbeltmærket.
- **10** blev anvendt til udvikling; **30** til held-out test.
- Mærkerne blev samlet til gold ved **union**, dvs. hvis en af de to annotatorer fandt et værktøj, kom det med.
- Agreement er dokumenteret som **F1 = 0,86** og **Cohen's κ = 0,90**.

**Hvorfor menneskelige labels?** Vi kan ikke blot måle »hvor meget matcheren finder«. Vi har brug for en reference for at måle fejl.

**Vigtig kritik:** Gold-labels er ikke ufejlbarlige. En union kan forbedre recall i referencen men også fastholde tvivlsomme fund. Derfor giver inter-annotator agreement information om labelkvaliteten.

Kilde: [`README.md`](https://github.com/Lucialululu/SkillRadar/blob/main/README.md). Tallene her er gengivet, ikke nyberegnet i denne notebook.

## 23. Precision, recall og F1 — konkrete formler

Ud fra sæt af (opslag, teknologi)-par:

- `TP`: systemet fandt et værktøj, der også er i gold.
- `FP`: systemet fandt et værktøj, der ikke er i gold.
- `FN`: gold indeholder et værktøj, som systemet ikke fandt.

\[
P = rac{TP}{TP+FP},\qquad R = rac{TP}{TP+FN},\qquad F1 = rac{2PR}{P+R}
\]

**Precision:** »Hvor stor en andel af de værktøjer, vi påstod, var korrekte?«

**Recall:** »Hvor stor en andel af de relevante værktøjer fandt vi?«

**F1:** Balancerer precision og recall, men kan skjule om en metode især fejler i én af dimensionerne.

I projektet er metrics micro-averaged over (item, tool)-par. Det betyder, at man summerer TP/FP/FN over dokumenter, inden man beregner metrics – ikke bare gennemsnitsberegner F1 pr. dokument.

In [ ]:
truth = {'Python','SQL','Docker','Databricks'}
predicted = {'Python','SQL','Go'}
tp = len(truth & predicted)
fp = len(predicted - truth)
fn = len(truth - predicted)
p = tp/(tp+fp) if tp+fp else 0
r = tp/(tp+fn) if tp+fn else 0
f1 = 2*p*r/(p+r) if p+r else 0
print('TP, FP, FN =', tp, fp, fn)
print(f'Precision = {p:.3f}; recall = {r:.3f}; F1 = {f1:.3f}')

**Output:** TP=2 (Python, SQL), FP=1 (Go), FN=2 (Docker, Databricks). Precision er 2/3, recall er 2/4, og F1 ligger mellem de to. Det er en **illustration**, ikke jeres projekttest.

## 24. Recall på O*NET-vokabular vs. recall på alle værktøjer

`evaluate.py` skelner mellem:

- `recall (vocab)`: Andelen af *gold-værktøjer, der er i jeres O*NET-katalog*, som bliver fundet.
- `recall (all)`: Andelen af **alle** gold-værktøjer, også OOV, som bliver fundet.

**Hvorfor?** En lukket keyword-matcher kan have fremragende recall på kataloget, men aldrig returnere ikke-katalogiserede teknologier. Derfor kan kun den første metric skjule den grundlæggende coverage-begrænsning.

Fra projektets README: Den frosne keyword-matcher opnår **precision 0,97**, **vocabulary recall 0,94**, men **recall på alle mærkede værktøjer 0,73** på 30 held-out postings. Ifølge README ligger omtrent **23 %** af de mærkede værktøjer uden for O*NET-vokabularet.

Det er ikke modstridende tal; de bruger *forskellige nævnere*.

In [ ]:
# Fiktivt eksempel på forskellig recall-definition
truth_vocab = {'Python', 'SQL', 'Docker'}
truth_oov = {'Databricks', 'NewPlatform'}
pred = {'Python', 'SQL', 'Docker'}
recall_vocab = len(pred & truth_vocab)/len(truth_vocab)
recall_all = len(pred & (truth_vocab|truth_oov))/len(truth_vocab|truth_oov)
print(f'Recall i vocabulary: {recall_vocab:.0%}')
print(f'Recall for alle værktøjer: {recall_all:.0%}')

**Output:** Matcheren finder alle kendte værktøjer (100 % vocabulary recall), men kun 3 af 5 faktiske værktøjer (60 % recall all). Det er et meget nyttigt eksamenseksempel på forskellen mellem en høj modelscore og en begrænset domænedækning.

## 25. Strict og lenient scoring

`evaluation/evaluate.py` har en standard **strict** evaluering og et opt-in flag `--lenient`:

- **Strict:** et fundet kanonisk tool skal matche gold direkte; OOV-teknologier krediteres som udgangspunkt ikke via løst navnematch.
- **Lenient:** ens navnevarianter for OOV kan tælle som samme værktøj efter normalisering, fx `Amazon SageMaker` vs. `SageMaker`.

**Hvorfor findes lenient?** En open-vocabulary model skal ikke straffes bare fordi den vælger et lidt længere produktnavn end menneskets label.

**Vigtigt:** Samme lenient-regel skal anvendes på alle metoder i sammenligningen. Ellers er sammenligningen unfair. Jeres README omtaler strict som officiel keywordscore og lenient ved de sammenlignede LLM-metoder.

**Begrænsning:** Løsere scoring kan også komme til at belønne navne, der lyder ens, men teknisk er forskellige. Projektet dokumenterer derfor manuel kontrol af lenient-par.

## 26. Resultater dokumenteret i README: hvad viste forsøgene?

| Måling på held-out postings | Dokumenteret resultat |
|---|---|
| Keyword precision | **0,97** |
| Keyword recall på vocabulary | **0,94** |
| Keyword recall på alle labels | **0,73** |
| Agreement mellem to labellere | **F1 0,86**, κ **0,90** |
| Hybrid recall på alle tools | **0,95** |
| Hybrid forskel i recall all vs. keyword | **+0,22**, 95 % interval **+0,15 til +0,29** |
| Hybrid precision | **0,97** ifølge README |
| Ekstra LLM-verificering alene | README angiver kun **1 afvist keyword-kandidat i 30 postings** |
| Speed | README angiver LLM-metoder ca. **18–22 sekunder pr. posting**, keyword i millisekunder |

**Fortolkning:** I dette testdesign blev den væsentligste gevinst ved Hybrid **højere dækningsgrad** – ikke at Gemini fandt et stort antal falske positive i keyword-kandidaterne. Hybrid kan finde OOV-technologier, som matcherens begrænsede ordforråd ikke kan.

**Forsigtighed:** Disse tal er fra README. De dokumenterer resultatet for den beskrevne held-out test og ikke nødvendigvis for alle CV-formater, jobroller eller den danske arbejdsmarkedsfordeling.

## 27. Hvorfor +0,22 ikke er det samme som +22 % relativ forbedring

`Recall all` går fra 0,73 til 0,95. Den **absolutte** ændring er `0,95 − 0,73 = 0,22`, altså **22 procentpoint**. Den relative stigning i recall er `0,22 / 0,73 ≈ 30 %`.

Til mundtlig eksamen er det vigtigt at bruge *procentpoint* ved forskellen mellem to andele.

Konfidensintervallet `+0,15 til +0,29` handler om den *parrede forbedring* på den evaluerede sample, ikke om et interval for hybridens egen recall.

In [ ]:
old, new = 0.73, 0.95
print(f'Absolut ændring: {(new-old)*100:.0f} procentpoint')
print(f'Relativ ændring: {(new-old)/old*100:.1f}%')

## 28. Bootstrap og sammenligning på de samme opslag

I `evaluate.py` anvender I **2.000 bootstrap-resamples over items** til 95 %-intervaller. Det er meningsfuldt, fordi værktøjer i samme jobopslag ikke er statistisk uafhængige. Hele opslag resamples sammen.

I laver også **paired bootstrap** for metodesammenligning, fordi begge modeller evalueres på de samme dokumenter.

**Hvorfor paired?** Hvis to metoder testes på forskellige, tilfældigt svære jobopslag, risikerer man at forveksle datavariation med metodeforbedring. Paired design fjerner meget af den variation.

**Alternativer:** Permutationstest for parrede forskelle, McNemars test på parrede fejl eller standard bootstrap af enkelte tools. Sidstnævnte ignorerer dog afhængigheden inden for et dokument.

**Til senere:** Vi gennemgår robusthedsstatistikken grundigere i Del 4. Her er den relevante pointe, at method comparison skal ske på *samme held-out testmateriale*.

In [ ]:
# Kun pædagogik: parret forbedring pr. dokument
paired = pd.DataFrame({'posting':['A','B','C','D','E'],
    'keyword_found':[2,1,3,1,2], 'hybrid_found':[3,2,3,2,3],
    'true_count':[3,2,3,2,3]})
paired['keyword_recall']=paired.keyword_found/paired.true_count
paired['hybrid_recall']=paired.hybrid_found/paired.true_count
paired['delta']=paired.hybrid_recall-paired.keyword_recall
display(paired)
print('Gns. paired forskel (kunstige data):',round(paired.delta.mean(),3))

### Output og metodevalg

Hver række er det samme opslag vurderet af to metoder. `delta` viser hvilken metode der fandt flest korrekte værktøjer i netop det opslag. Den gennemsnitlige forskel her er **ikke** projektets dokumenterede +0,22 og er ikke et bootstrap-estimat; den illustrerer blot parringen.

## 29. Hvorfor valgte I ikke embedding-linking i det frosne resultat?

Ifølge [`evaluation/llm_frozen_settings.txt`](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/llm_frozen_settings.txt) er indstillingerne fra den frosne evaluering 4. november 2026:

```ini
SKILLRADAR_LLM_MODEL=gemini-3.5-flash-lite
SKILLRADAR_EMBED_MODEL=gemini-embedding-001
SKILLRADAR_LINK_THRESHOLD=1.0
SKILLRADAR_FEWSHOT=post_003,post_005
```

Den samme fil forklarer, at embedding-linking på dev-sættet sænkede performance, mens threshold 1.0 (deaktiveret) gav bedst F1 (all) for både LLMOnly og Hybrid.

**Nuance:** Kodefilens *default settings* kan godt afvige fra de *frosne evalueringsindstillinger*. Hvis man blot kører uden `.env`, må man ikke automatisk påstå, at man reproducerer de publicerede testtal.

**Eksamen:** Forklar forskellen mellem `default`, `dev-tunet` og `held-out test`.

## 30. Hvad betyder hastighed for valget af produktionsløsning?

README beskriver ca. **18–22 sekunder pr. posting** for LLM-modellerne, mod millisekunder for keyword. Når der er tusinder af postings, skalerer latenstiden voldsomt, især med eksterne API'er.

**Vigtig arkitekturdetalje:** Jobopslagene bliver allerede behandlet offline. Appen behøver ikke genkøre LLM på alle opslag ved CV-upload. Men at køre LLM på brugerens CV kan stadig introducere forsinkelse, omkostning, privatlivshensyn og flere potentielle fejlkilder.

**To rimelige perspektiver:**
- Keyword i appen er et godt valg for hastighed, reproducerbarhed og privatliv.
- Hybrid viser klart bedre recall i evalueringen og kan være relevant i en senere version, hvis man kan håndtere omkostninger, latency og samtykke.

Det er ikke nok at sige at én metode er »bedst«. Man skal specificere om kriteriet er accuracy, recall all, præcision, hastighed eller brugeroplevelse.

## 31. Human in the loop: hvorfor brugeren tjekker forslag

I `app.py` viser `step_check()` fundne værktøjer sammen med evidence. Brugeren kan:

1. **Fjerne** fejlagtige fund ved at fjerne afkrydsningen.
2. **Tilføje** en teknologi fra det eksisterende katalog, hvis matcheren overså den.

`step_report()` bruger kun de beholdte og manuelt tilføjede værktøjer.

**Hvad løser det?** Fejl ved keyword-matching kan opdages af brugeren; det gør den konkrete rapport mere personligt korrekt.

**Hvad løser det ikke?** Brugeren kan ikke automatisk garantere, at alle værktøjer er korrekt forstået. Det løser heller ikke OOV-begrænsningen for tilføjelse fra det eksisterende katalog.

**Til eksamen:** Undgå at sige »brugeren retter modellen«. Brugeren retter **sin aktuelle sessions værktøjsliste**, ikke modelparametrene eller O*NET-kataloget.

## 32. Kodeeksempel: fra CV til kandidatværktøjer og brugerrettelse

Følgende er en undervisningssimulation af webappens trin. Den anvender vores forenklede `demo_explain()`, ikke produktionsmatcheren.

In [ ]:
fake_cv = 'I know Python and Power BI, and I used Go in a backend course.'
found = set(demo_explain(fake_cv).values())
print('Automatisk fund:',sorted(found))
removed = {'Go programming language'}       # simulér brugerens vurdering
added = {'Structured query language SQL'}    # simulér brugerens tilføjelse
corrected = (found-removed)|added
print('Efter brugerrettelse:',sorted(corrected))

**Output:** Forskellen mellem rå automatiske fund og den bekræftede værktøjsliste bliver tydelig. I kan dermed fortælle eksaminator, præcis hvor menneskelig korrektion indgår, og at ranking ikke baseres på den ukorrigerede liste.

## 33. Typiske metodiske fejl, I skal kunne diskutere

**False positive fra tvetydige navne:** `Go` som imperativ, `Google` som arbejdsgiver.

**False negative fra OOV:** Et nyt værktøj findes i teksten, men ikke i O*NET-kataloget.

**False negative fra formulering:** En synonym variant er ikke i alias-tabellen.

**Forkert »skill«-definition:** `machine learning` er en generel kompetence og tælles ikke som et navngivet softwareværktøj i appens scope.

**Wrong linking:** Embedding placerer et produkt under et andet nærtliggende produkt.

**LLM-hallucination:** Modellen tilføjer typiske teknologier uden tekstbevis; grounding begrænser dette.

**Label bias:** Menneskelige gold-labels er ufuldstændige eller forskellige.

**Distribution shift:** Evalueringsmateriale med Data Scientist-opslag siger ikke nødvendigvis alt om fx UX Designer-CV'er.

**Privacy:** Eksterne API'er og diskcache kræver særskilt håndtering, navnlig for CV'er.

## 34. Alternativesammenligning til eksamen

| Metode | Fordele | Ulemper | God brug |
|---|---|---|---|
| Eksakt stringmatch | Meget hurtig, transparent | Meget lav synonymdækning | Enkel baseline |
| O*NET + aliases | Hurtig, ensartede navne, kan testes | OOV og tvetydige navne | Hovedapp og offline preprocessing |
| Regex + kontekstregler | God kontrol over kendte fejl | Mange specialregler | Smalle, veldefinerede problemer |
| spaCy NER finetunet | Lærer kontekst | Kræver annoterede eksempler og træning | Hvis mange labels er til rådighed |
| LLMOnly | Open vocabulary, sprogforståelse | Pris/latency/hallucination/privatliv | Offline evaluerings- eller fremtidig extraction |
| Hybrid | Bevarer baseline, forbedrer dækning | Mere kompleks, API-afhængig | Hvor recall all er vigtig nok til at betale prisen |
| Embedding-linker | Kan finde nye skrivemåder | Semantisk nærhed ≠ samme produkt | Kun med stærk disambiguation/evaluering |

**Valget kan begrundes i jeres faktiske målinger:** Der er høj precision for keyword på det evaluerede materiale, men lavere recall all på grund af OOV. Hybrid giver bedre dækning, men koster tid og integration.

## 35. Eksamensspørgsmål med svar

**1. Hvorfor bruger I samme matcher på CV og posting?**  
Så begge repræsenteres ved samme kanoniske O*NET-navne og kan sammenlignes. Hvis man brugte forskellige ordforråd, kunne identiske teknologier få forskellige identiteter.

**2. Hvorfor er `set` vigtigt?**  
Det forhindrer, at et værktøj tælles flere gange i samme tekst.

**3. Kan keyword-matcheren forstå kontekst?**  
Kun i begrænset grad gennem ordgrenser, case og manuelle regler – ikke fuld semantik.

**4. Hvad er jeres egentlige taxonomy-mapping?**  
Genkendte aliaser mappes til O*NET's kanoniske navne; ikke en generel relationel skills-ontologi.

**5. Hvorfor har I ikke bare valgt LLMOnly?**  
Højere modelkompleksitet, svartid, omkostning og privatliv; produktion og forskningssammenligning har forskellige krav.

**6. Hvorfor HybridVerify?**  
For at se, om gevinsten ligger i at verificere eksisterende fund eller tilføje oversete værktøjer.

**7. Hvad viser 0,94 vs. 0,73 recall?**  
Høj recall på kendte katalogværktøjer, men markant lavere på alle værktøjer inklusive OOV.

**8. Hvad betyder +0,22?**  
22 procentpoint forbedring i recall all – ikke 22 % relativ stigning.

**9. Hvorfor slå embedding-linking fra?**  
Udviklingstest viste dårligere resultater ved embeddings-linking; beslægtede værktøjer blev forvekslet.

**10. Hvad er »grounding«?**  
At kontrollere om det foreslåede værktøj har tekstligt belæg i originalen, så frie LLM-påstande kan afvises.

**11. Hvilken data brugte I til at vælge few-shot?**  
Kun dev-eksempler, ikke test.

**12. Hvordan kan I bevise en metodeforbedring?**  
Samme held-out dokumenter, samme gold og scoring, paired bootstrap af forskellen.

**13. Har I trænet Gemini?**  
Nej, I bruger en prætrænet model via prompts og postprocessing.

**14. Hvad ville du ændre i næste version?**  
Eksempelvis kontrollere open-vocabulary-fund i human-in-the-loop, måle CV-resultater i større sample og undersøge latency/privacy ved en hybridproduktion.

## 36. Et mundtligt svar på ca. to minutter

> »Vores primære mål i matching-delen var at finde navngivne teknologier i både jobopslag og CV'er, og normalisere dem til de samme navne. Vi tog udgangspunkt i O*NET's softwareværktøjer markeret som Hot Technology eller In Demand. Fordi O*NET's navne ofte er længere end almindelige formuleringer, lavede vi en keyword-matcher, som genererer aliaser, fx Power BI fra Microsoft Power BI og SQL fra det lange O*NET-navn. Matcheren bruger tokenisering, n-grams, case-sensitive regler og nogle manuelt fjernede tvetydige aliaser.
>
> Appen bruger den regelbaserede matcher, fordi den er hurtig og kan køre lokalt. Brugeren får vist tekstbevis for de fundne værktøjer og kan rette dem før rapporten. Som forskning har vi så testet, om de ekstra normaliseringstrin faktisk forbedrer resultaterne, ved en ablation med exact, vendor, acronym og full.
>
> Vi sammenlignede også keyword-modellen med en prætrænet LLM og en hybrid. LLM'en skal angive citater, og vi har grounding for at begrænse opfundne fund. Hybrid bruger keyword-kandidater og lader LLM verificere og tilføje oversete værktøjer. En verificering-uden-tilføjelse-variant viste, hvor meget gevinsten kommer fra nye fund.
>
> På vores dokumenterede held-out posting-test havde keyword høj precision og høj recall på værktøjer i O*NET, men lavere recall på alle værktøjer, fordi en del lå uden for kataloget. Hybrid forbedrede recall på alle værktøjer fra 0,73 til 0,95 med uændret precision omkring 0,97 ifølge vores README. Til gengæld er LLM meget langsommere og har andre omkostninger og persondatahensyn. Vi har ikke finetunet modellen; vi har brugt prompting, normalisering og en kontrolleret evaluering.«

## 37. Prøv selv: lille afsluttende eksamensøvelse

Forestil dig følgende jobopslag:

> `We are hiring a data scientist. You will build dashboards with Power BI, write SQL queries, and train models in Python. Experience with Databricks is a plus.`

**Opgave A:** Hvilke tools bør jeres definition godkende?

**Opgave B:** Hvilke af værktøjerne kræver aliaser fra lange O*NET-navne?

**Opgave C:** Hvad sker der, hvis `Databricks` ikke er i jeres O*NET-ordforråd, og metoden er KeywordMatcher vs. LLMOnly?

**Opgave D:** Hvad vil du kræve af LLM'ens `quote` for hvert fund?

**Opgave E:** Hvilken test kan isolere, om en hybridmetode især bidrager med verification eller additions?

**Vejledende svar:** A: Power BI, SQL, Python, Databricks. B: Især Power BI og SQL, afhængigt af katalogets fulde navne. C: Keyword kan ikke returnere OOV, LLM kan foreslå det som `OOV:Databricks` efter grounding/linking. D: En kort ordret tekstpassage fra opslaget, hvor værktøjet faktisk står. E: Sammenlign `Hybrid`, `HybridVerify` og keyword på de samme gold-labeled dokumenter.

## 38. Kildeliste og hvad vi ikke har genberegnet

**Direkte kildekode (SkillRadar GitHub, læst 5. november 2026):**
- [`skillradar/matcher.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/matcher.py)
- [`skillradar/variants.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/variants.py)
- [`skillradar/llm_matcher.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/llm_matcher.py)
- [`evaluation/evaluate.py`](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/evaluate.py)
- [`evaluation/llm_frozen_settings.txt`](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/llm_frozen_settings.txt)
- [`app.py`](https://github.com/Lucialululu/SkillRadar/blob/main/app.py)
- [`README.md`](https://github.com/Lucialululu/SkillRadar/blob/main/README.md)

**Afgrænsning:** De kørbare illustrationer benytter små, syntetiske tekster og forenklede ordbøger for at kunne køres uden O*NET-data og Gemini. Den reelle algoritme forklares med direkte kodeuddrag, men den er ikke reproduceret af eksempelcellerne. De målte precision/recall- og tidsresultater er gengivet fra README og de dokumenterede frosne indstillinger; de er ikke re-evalueret her. Det betyder også, at denne notebook ikke alene dokumenterer alle CV-evalueringsresultater.

**Efter Del 3:** Del 4 undersøger demand ranking, Wilson-intervaller, coverage, Spearman, bootstrap, senioritetsforskelle og statistiske begrænsninger.